In [ ]:
import re
import html
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

config


In [ ]:
RAW_PATH = Path("../data/raw/upwork-jobs.csv")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
WEIGHTS = {
    "required": 0.50,      # required skills covered
    "preferred": 0.10,     # preferred skills covered
    "experience": 0.15,    # years of experience vs. level asked
    "availability": 0.15,  # availability
    "quality": 0.10,       # verified + portfolio + complete profile
}
MIN_SCORE = 30             # minimum score shown
MAX_RESULTS = 50           # max items in the list
PAGE_SIZE = 10             # items per page
MAX_ACTIVE_PROJECTS = 3    # DEVSYNC-US-45
LIMIT_PENALTY = 0.3        # availability is multiplied by this at the project limit
PORTFOLIO_CAP = 5          # 5+ portfolio items = full points
QUALITY_W = {"verified": 0.4, "portfolio": 0.3, "complete": 0.3}


In [ ]:
REQUIRED_TOP_N = 3         # first 3 skills = required, the rest = preferred
DEFAULT_REQUIRED_YEARS = 2

In [ ]:
# ---------- Simulation settings ----------
SEED = 42
N_DEVELOPERS = 2000
N_TEAMS = 300
BASE_DATE = pd.Timestamp("2024-02-24")
rng = np.random.default_rng(SEED)

assert abs(sum(WEIGHTS.values()) - 1) < 1e-9, "weights must sum to 1"
print("Config OK")

load raw data

In [ ]:
df_raw = pd.read_csv(RAW_PATH)
projects = df_raw.copy()

print("Shape:", projects.shape)
display(projects.head(3))

parsing functions

In [ ]:
# Category and Skills are embedded at the END of the description.
# We cut from the LAST "Posted On:" because some descriptions contain "Category:" in their body.
END_MARKERS = r"(?:Skills:|Location Requirement:|Country:|click to apply|$)"

def get_tail(text):
    i = text.rfind("Posted On:")
    return text[i:] if i >= 0 else ""

def parse_category(tail):
    m = re.search(r"Category:(.*?)" + END_MARKERS, tail, re.S)
    return re.sub(r"\s+", " ", m.group(1)).strip() if m else None

def parse_skills(tail):
    # Skills appear twice in the text, so take the first block
    m = re.search(r"Skills:(.*?)Skills:", tail, re.S)
    if not m:
        m = re.search(r"Skills:(.*?)(?:Location Requirement:|Country:|click to apply|$)", tail, re.S)
    if not m:
        return []
    skills, seen = [], set()
    for s in m.group(1).split(","):
        s = re.sub(r"\s+", " ", s).strip()
        if s and s.lower() not in seen:      # remove duplicates, keep order
            seen.add(s.lower())
            skills.append(s)
    return skills

def parse_body(text):
    i = text.rfind("Posted On:")
    body = text[:i] if i >= 0 else text
    # remove the Budget / Hourly Range at the end of the description
    body = re.sub(r"(Budget:\s*\$[\d,\.]+|Hourly Range:\s*\$[\d\.,]+\s*-\s*\$[\d\.,]+)\s*$", "", body)
    body = re.sub(r"https?://\S+|\S+@\S+", " ", body)    # remove links and emails
    return re.sub(r"\s+", " ", body).strip()

def parse_posting(raw_text):
    text = html.unescape(raw_text)        # &amp; -> &
    tail = get_tail(text)
    return {
        "category": parse_category(tail),
        "skills": parse_skills(tail),
        "description_body": parse_body(text),
    }

print("Parsing functions ready")

apply parsing

In [ ]:
parsed = pd.DataFrame([parse_posting(t) for t in projects["description"]], index=projects.index)
projects = projects.join(parsed)
projects["n_skills"] = projects["skills"].str.len()

print("Projects with category:", projects["category"].notna().sum())
print("Projects with >=1 skill:", (projects["n_skills"] > 0).sum(), f"({(projects['n_skills'] > 0).mean():.1%})")
print("Unique categories:", projects["category"].nunique())
display(projects[["title", "category", "skills"]].head(5))

remove duplicates

In [ ]:
# Same job reposted with a different link
projects["title_key"] = projects["title"].str.lower().str.strip()
before = len(projects)
projects = projects.drop_duplicates(subset=["title_key", "description_body"]).copy()
print(f"Removed duplicates: {before - len(projects)}  |  Remaining: {len(projects)}")

budget normalization

In [ ]:
# Payment type + outlier capping
projects["budget_type"] = projects["is_hourly"].map({True: "hourly", False: "fixed"}).fillna("unspecified")

# Cap extreme budgets at the 99th percentile (raw max is 1,000,000)
cap = projects["budget"].quantile(0.99)
projects["budget_capped"] = projects["budget"].clip(upper=cap)
print(f"Budget cap (p99): {cap:,.0f}")

# Missing values follow the payment type (not lost data)
display(projects.groupby("budget_type")[["budget", "hourly_low", "hourly_high"]].count())

required experience (years)

In [ ]:
YEARS_RE = re.compile(r"(\d{1,2})\s*\+?\s*(?:years?|yrs?)", re.I)
LEVELS = [
    (re.compile(r"\b(expert|senior|advanced)\b", re.I), 5),
    (re.compile(r"\b(intermediate|mid[- ]level)\b", re.I), 3),
    (re.compile(r"\b(entry[- ]level|junior|beginner)\b", re.I), 1),
]

def extract_required_years(title, body):
    text = f"{title} {body}"
    m = YEARS_RE.search(text)
    if m:
        return min(int(m.group(1)), 10)
    for rx, years in LEVELS:
        if rx.search(text):
            return years
    return DEFAULT_REQUIRED_YEARS

projects["required_years"] = [
    extract_required_years(t, b) for t, b in zip(projects["title"], projects["description_body"])
]
print(projects["required_years"].value_counts().sort_index())

tech filter + required/preferred split

In [ ]:
TECH_CATEGORIES = [
    "Full Stack Development", "Front-End Development", "Back-End Development",
    "Mobile App Development", "Ecommerce Website Development", "CMS Development",
    "Scripting & Automation", "Desktop Software Development", "Business Applications Development",
    "Database Development", "Database Administration", "Firmware Development",
    "Video Game Development", "Mobile Game Development", "AR/VR Development",
    "DevOps Engineering", "Cloud Engineering",
    "Data Analytics", "Data Engineering", "Data Mining",
    "Machine Learning", "Generative AI Modeling", "AI Integration", "AI Chatbot Development",
    "Manual Testing", "Automation Testing",
    "Blockchain & NFT Development", "Crypto Wallet Development",
]

is_tech = projects["category"].isin(TECH_CATEGORIES) & (projects["n_skills"] > 0)
projects_tech = projects[is_tech].copy()

# First REQUIRED_TOP_N skills (Upwork order) = required, the rest = preferred
projects_tech["required_skills"] = projects_tech["skills"].map(lambda s: s[:REQUIRED_TOP_N])
projects_tech["preferred_skills"] = projects_tech["skills"].map(lambda s: s[REQUIRED_TOP_N:])

# project_id from the link
pid = projects_tech["link"].str.extract(r"_%7E(0\w+)")[0]
projects_tech["project_id"] = pid.fillna(pd.Series(projects_tech.index.astype(str), index=projects_tech.index).radd("row-"))

print("Tech projects:", len(projects_tech), "of", len(projects))
display(projects_tech[["project_id", "title", "category", "required_skills", "preferred_skills", "required_years"]].head(5))

save cleaned data

In [ ]:
def save_df(frame, name):
    try:
        frame.to_parquet(PROCESSED_DIR / f"{name}.parquet")
        print("Saved:", PROCESSED_DIR / f"{name}.parquet")
    except ImportError:
        frame.to_pickle(PROCESSED_DIR / f"{name}.pkl")
        print("Saved (pickle):", PROCESSED_DIR / f"{name}.pkl")

KEEP_COLS = ["project_id", "title", "link", "published_date", "category", "skills",
"required_skills", "preferred_skills", "required_years", "description_body",
"budget_type", "budget", "budget_capped", "hourly_low", "hourly_high", "country"]
save_df(projects_tech[KEEP_COLS], "projects_clean")

skill pools per category

In [ ]:
# Most common skills in each category (realistic source for developer skills)
cat_counts = projects_tech["category"].value_counts()
dev_cats = [c for c in cat_counts.index if cat_counts[c] >= 100]

cat_skill_pool = {}
for cat in dev_cats:
    counter = Counter(s.lower() for lst in projects_tech.loc[projects_tech["category"] == cat, "skills"] for s in lst)
    top = counter.most_common(40)
    names, freq = zip(*top)
    p = np.array(freq, dtype=float)
    cat_skill_pool[cat] = (list(names), p / p.sum())

cat_probs = cat_counts[dev_cats].to_numpy(dtype=float)
cat_probs = cat_probs / cat_probs.sum()

country_counts = projects["country"].dropna().value_counts().head(20)
country_probs = (country_counts / country_counts.sum()).to_numpy()

print("Categories used:", len(dev_cats))
print({c: len(cat_skill_pool[c][0]) for c in dev_cats[:3]})

generate developers

In [ ]:
dev_rows = []
for i in range(1, N_DEVELOPERS + 1):
    main_cat = str(rng.choice(dev_cats, p=cat_probs))
    pool, p = cat_skill_pool[main_cat]
    skills = {str(s) for s in rng.choice(pool, size=min(int(rng.integers(4, 9)), len(pool)), replace=False, p=p)}

    # 2 extra skills from another field (realistic variety)
    other = str(rng.choice(dev_cats, p=cat_probs))
    o_pool, o_p = cat_skill_pool[other]
    skills |= {str(s) for s in rng.choice(o_pool, size=2, replace=False, p=o_p)}

    dev_rows.append({
        "developer_id": f"D{i:04d}",
        "name": f"Developer {i:04d}",
        "gender": str(rng.choice(["F", "M"])),
        "country": str(rng.choice(country_counts.index, p=country_probs)),
        "main_category": main_cat,
        "skills_list": sorted(skills),
        "years": int(np.clip(rng.gamma(2.0, 2.0), 0, 15)),
        "availability": str(rng.choice(["Available", "Unavailable"], p=[0.75, 0.25])),
        "active_projects": int(rng.choice([0, 1, 2, 3, 4], p=[0.35, 0.30, 0.20, 0.10, 0.05])),
        "verified": bool(rng.random() < 0.4),
        "portfolio_items": int(rng.poisson(2.5)),
        "completion_pct": int(rng.choice([60, 70, 80, 90, 100], p=[0.05, 0.10, 0.15, 0.20, 0.50])),
        "appear_in_recs": bool(rng.random() < 0.9),
        "account_status": str(rng.choice(["Active", "Suspended", "Deleted"], p=[0.94, 0.03, 0.03])),
        "updated_at": BASE_DATE - pd.Timedelta(days=int(rng.integers(0, 120))),
    })

devs = pd.DataFrame(dev_rows)
print(devs.shape)
display(devs.head(3))
print(devs["account_status"].value_counts().to_dict(), "| unavailable:", (devs["availability"] == "Unavailable").mean().round(2))

generate teams and members

In [ ]:
active_dev_ids = devs.loc[devs["account_status"] == "Active", "developer_id"].to_numpy()

team_rows, member_rows = [], []
for t in range(1, N_TEAMS + 1):
    team_id = f"T{t:03d}"
    members = rng.choice(active_dev_ids, size=int(rng.integers(2, 6)), replace=False)
    team_rows.append({
        "team_id": team_id,
        "name": f"Team {t:03d}",
        "team_status": str(rng.choice(["Active", "Inactive"], p=[0.9, 0.1])),
        "appear_in_recs": bool(rng.random() < 0.9),
        "updated_at": BASE_DATE - pd.Timedelta(days=int(rng.integers(0, 120))),
    })
    for j, dev_id in enumerate(members):
        member_rows.append({
            "team_id": team_id,
            "developer_id": str(dev_id),
            "role": "Lead" if j == 0 else "Member",
            "member_status": "Active" if (j == 0 or rng.random() < 0.9) else "Left",
        })

teams = pd.DataFrame(team_rows)
team_members = pd.DataFrame(member_rows)
print("Teams:", len(teams), "| Members rows:", len(team_members))
display(team_members.head(5))

project settings and existing applications/invitations

In [ ]:
# Project settings that are not in the original data: owner, who is accepted, status
n = len(projects_tech)
projects_tech["status"] = "Open"
projects_tech["accepts"] = rng.choice(["both", "individuals", "teams"], size=n, p=[0.6, 0.2, 0.2])
owners = np.array([f"C{k:04d}" for k in rng.integers(1, 500, size=n)], dtype=object)
dev_owner_mask = rng.random(n) < 0.02          # 2% of projects are owned by a developer (to test the owner rule)
owners[dev_owner_mask] = rng.choice(devs["developer_id"].to_numpy(), size=int(dev_owner_mask.sum()))
projects_tech["owner_user_id"] = owners

# Existing Applied / Invited records
all_candidate_ids = np.concatenate([devs["developer_id"].to_numpy(), teams["team_id"].to_numpy()])
inter_rows = []
for pid_ in rng.choice(projects_tech["project_id"].to_numpy(), size=400, replace=False):
    for cand_id in rng.choice(all_candidate_ids, size=int(rng.integers(1, 4)), replace=False):
        inter_rows.append({"project_id": pid_, "candidate_id": str(cand_id),
        "status": str(rng.choice(["Applied", "Invited"]))})
interactions = pd.DataFrame(inter_rows).drop_duplicates(["project_id", "candidate_id"])
print("Interactions:", len(interactions), interactions["status"].value_counts().to_dict())

save simulated tables

In [ ]:
for name, frame in [("developers", devs), ("teams", teams), ("team_members", team_members),
                    ("interactions", interactions)]:
    save_df(frame, name)

unified candidate table (developers + teams)

In [ ]:
CAND_COLS = ["candidate_id", "candidate_type", "name", "skills", "years", "avail_factor",
"unavailable", "at_limit", "verified", "portfolio", "completion_pct",
"updated_at", "eligible"]

def build_dev_features(devs_df):
    d = devs_df.copy()
    d["skills"] = d["skills_list"].map(lambda lst: frozenset(s.lower() for s in lst))
    d["unavailable"] = d["availability"].eq("Unavailable")
    d["at_limit"] = d["active_projects"] >= MAX_ACTIVE_PROJECTS
    d["avail_factor"] = np.where(d["unavailable"], 0.0, 1.0)
    d.loc[d["at_limit"], "avail_factor"] *= LIMIT_PENALTY      # a signal, not a lock
    d["verified"] = d["verified"].astype(float)
    d["portfolio"] = d["portfolio_items"].astype(float)
    return d

def build_candidate_table(devs_df, teams_df, members_df):
    d = build_dev_features(devs_df)

    # A developer appears if: Active + complete profile + "Appear in recommendations" is on
    dev_c = pd.DataFrame({
        "candidate_id": d["developer_id"], "candidate_type": "Developer", "name": d["name"],
        "skills": d["skills"], "years": d["years"].astype(float), "avail_factor": d["avail_factor"],
        "unavailable": d["unavailable"], "at_limit": d["at_limit"], "verified": d["verified"],
        "portfolio": d["portfolio"], "completion_pct": d["completion_pct"].astype(float),
        "updated_at": d["updated_at"],
        "eligible": d["account_status"].eq("Active") & (d["completion_pct"] >= 90) & d["appear_in_recs"],
    })[CAND_COLS]
    if len(teams_df) == 0:
        return dev_c.reset_index(drop=True)

    # A team combines the skills of its ACTIVE members
    active = members_df[members_df["member_status"] == "Active"].merge(
        d[d["account_status"] == "Active"], on="developer_id")
    g = active.groupby("team_id")
    agg = pd.DataFrame({
        "skills": g["skills"].agg(lambda s: frozenset().union(*s)),
        "years": g["years"].mean(),
        "avail_factor": g["avail_factor"].mean(),
        "unavailable": g["unavailable"].all(),
        "at_limit": g["at_limit"].all(),
        "verified": g["verified"].mean(),
        "portfolio": g["portfolio"].mean(),
        "completion_pct": g["completion_pct"].mean(),
        "members_updated": g["updated_at"].max(),
        "n_active": g.size(),
    }).reset_index()

    t = teams_df.merge(agg, on="team_id", how="inner")
    team_c = pd.DataFrame({
        "candidate_id": t["team_id"], "candidate_type": "Team", "name": t["name"],
        "skills": t["skills"], "years": t["years"], "avail_factor": t["avail_factor"],
        "unavailable": t["unavailable"], "at_limit": t["at_limit"], "verified": t["verified"],
        "portfolio": t["portfolio"], "completion_pct": t["completion_pct"],
        "updated_at": t[["updated_at", "members_updated"]].max(axis=1),
        "eligible": t["team_status"].eq("Active") & t["appear_in_recs"] & (t["n_active"] >= 1),
    })[CAND_COLS]
    return pd.concat([dev_c, team_c], ignore_index=True)

candidates = build_candidate_table(devs, teams, team_members)
print(candidates.shape, "| eligible:", int(candidates["eligible"].sum()))
print(candidates.groupby("candidate_type")["eligible"].agg(["size", "sum"]))

score function (0-100)

In [ ]:
# The function reads ONLY: skills, years, avail_factor, verified, portfolio, completion_pct
# So name / photo / gender / country are never used (fairness).
def score_candidates(project, cand, weights=WEIGHTS):
    req = {s.lower() for s in project["required_skills"]}
    pref = {s.lower() for s in project["preferred_skills"]}
    w = dict(weights)
    if not pref:
        w["preferred"] = 0.0          # no preferred skills -> redistribute the weight
    total_w = sum(w.values())

    factors = pd.DataFrame(index=cand.index)
    factors["required"] = cand["skills"].map(lambda s: len(req & s) / len(req))
    factors["preferred"] = cand["skills"].map(lambda s: len(pref & s) / len(pref)) if pref else 0.0
    factors["experience"] = (cand["years"] / max(project["required_years"], 1)).clip(upper=1.0)
    factors["availability"] = cand["avail_factor"]
    factors["quality"] = (
        QUALITY_W["verified"] * cand["verified"]
        + QUALITY_W["portfolio"] * (cand["portfolio"] / PORTFOLIO_CAP).clip(upper=1.0)
        + QUALITY_W["complete"] * cand["completion_pct"] / 100.0
    )

    points = factors.mul(pd.Series(w)) / total_w * 100       # contribution of each factor
    out = cand.copy()
    for col in factors.columns:
        out[f"factor_{col}"] = factors[col]
        out[f"points_{col}"] = points[col]
    out["score"] = points.sum(axis=1).round().astype(int)
    return out

access check (Step 2 of the main flow)

In [ ]:
MSG_NOT_OWNER = "You are not allowed to view the matches of this project"
MSG_NOT_OPEN = "Matches are available only for open projects"
MSG_NO_SKILLS = "Add required skills to your project to see matches"

def check_project_access(project, requester_id):
    if project["owner_user_id"] != requester_id:
        return MSG_NOT_OWNER
    if project["status"] != "Open":
        return MSG_NOT_OPEN
    if len(project["required_skills"]) == 0:
        return MSG_NO_SKILLS
    return None      # None = allowed

full ranking

In [ ]:
def rank_matches(project, cand, interactions_df, weights=WEIGHTS):
    pool = cand[cand["eligible"]]
    pool = pool[pool["candidate_id"] != project["owner_user_id"]]          # the owner never appears

    if project["accepts"] == "individuals":                                 # respect project type (US-27)
        pool = pool[pool["candidate_type"] == "Developer"]
    elif project["accepts"] == "teams":
        pool = pool[pool["candidate_type"] == "Team"]

    scored = score_candidates(project, pool, weights)

    # Design decision: must cover at least ONE required skill.
    # Without it, a candidate with zero required skills can still reach 40-50
    # from experience + availability + quality and pass the threshold of 30.
    scored = scored[scored["factor_required"] > 0]
    scored = scored[scored["score"] >= MIN_SCORE]

    scored = scored.sort_values(["score", "updated_at"], ascending=[False, False]).head(MAX_RESULTS).copy()

    # Matching skills (max 5) and missing skills (max 3)
    req_display = list(project["required_skills"])
    scored["matched_skills"] = scored["skills"].map(lambda s: [r for r in req_display if r.lower() in s][:5])
    scored["missing_skills"] = scored["skills"].map(lambda s: [r for r in req_display if r.lower() not in s][:3])

    # Marks: Applied / Invited / Verified / Unavailable / At the limit
    status_map = (interactions_df[interactions_df["project_id"] == project["project_id"]]
.set_index("candidate_id")["status"].to_dict())
    scored["interaction"] = scored["candidate_id"].map(status_map)

    def make_marks(row):
        marks = []
        if isinstance(row["interaction"], str):
            marks.append(row["interaction"])
        if row["verified"] >= 1.0:
            marks.append("Verified")
        if row["unavailable"]:
            marks.append("Unavailable")
        if row["at_limit"]:
            marks.append("At the limit of projects")
        return marks

    scored["marks"] = scored.apply(make_marks, axis=1) if len(scored) else []
    scored["can_invite"] = scored["interaction"].isna()           # Invite button disabled for Applied/Invited
    scored["rank"] = np.arange(1, len(scored) + 1)
    scored["page"] = (scored["rank"] - 1) // PAGE_SIZE + 1
    return scored.reset_index(drop=True)

SHOW_COLS = ["rank", "candidate_id", "candidate_type", "score", "matched_skills", "missing_skills", "marks", "can_invite"]

type filter and pagination

In [ ]:
def filter_by_type(matches, choice="All"):
    if choice == "Developers":
        out = matches[matches["candidate_type"] == "Developer"]
    elif choice == "Teams":
        out = matches[matches["candidate_type"] == "Team"]
    else:
        out = matches
    return out.reset_index(drop=True).assign(rank=lambda x: np.arange(1, len(x) + 1))

def get_page(matches, page=1):
    start = (page - 1) * PAGE_SIZE
    return matches.iloc[start:start + PAGE_SIZE]

def available_filters(matches):
    # an option with no results is disabled
    types = set(matches["candidate_type"])
    return {"All": len(matches) > 0, "Developers": "Developer" in types, "Teams": "Team" in types}

try it on a real project

In [ ]:
def skills_in_pool(row):
    pool = cat_skill_pool.get(row["category"], ([], None))[0]
    return all(s.lower() in pool for s in row["required_skills"])

candidates_demo = projects_tech[
    (projects_tech["category"] == "Full Stack Development")
    & (projects_tech["n_skills"] >= 5)
    & (projects_tech["accepts"] == "both")
]
demo = candidates_demo[candidates_demo.apply(skills_in_pool, axis=1)].iloc[0]

print("Project:", demo["title"])
print("Required :", demo["required_skills"])
print("Preferred:", demo["preferred_skills"])
print("Years    :", demo["required_years"], "| Owner:", demo["owner_user_id"], "| Accepts:", demo["accepts"])
print("Access check:", check_project_access(demo, demo["owner_user_id"]))

matches = rank_matches(demo, candidates, interactions)
print("\nTotal matches:", len(matches), "| max score:", matches["score"].max(), "| min score:", matches["score"].min())
display(get_page(matches, 1)[SHOW_COLS])

try the filter and pages

In [ ]:
print("Filters available:", available_filters(matches))
teams_only = filter_by_type(matches, "Teams")
print("Teams only:", len(teams_only), "| page 1 size:", len(get_page(teams_only, 1)))
display(get_page(teams_only, 1)[SHOW_COLS].head(3))

# الجزء الثاني: شرح النتيجة وحالة التحديث وتجهيز استجابة الـ API


فحص الاعتماديات (يجب تشغيل الجزء الأول أولًا في نفس الـ kernel)


In [ ]:
import json
import math
from datetime import datetime, timedelta

import pandas as pd

# هذه الأسماء معرّفة في الجزء الأول (part 1)
_REQUIRED = ["get_page", "PAGE_SIZE", "available_filters", "rank_matches",
             "skills_in_pool", "projects_tech", "candidates", "interactions"]
_missing = [n for n in _REQUIRED if n not in globals()]
if _missing:
    raise NameError(
        "هذه العناصر غير معرّفة: " + ", ".join(_missing)
        + " — شغّل الجزء الأول (part 1) أولًا في نفس الـ kernel."
    )

# ---------- دوال مساعدة آمنة ----------
def _py(v):
    """تحويل أنواع numpy/pandas إلى أنواع بايثون العادية (لتجنب أخطاء JSON)."""
    if hasattr(v, "item") and not isinstance(v, (list, tuple, dict, str)):
        try:
            return v.item()
        except Exception:
            pass
    return v

def _num(v, default=0):
    """تحويل آمن إلى int (يتعامل مع NaN و None)."""
    try:
        if v is None or pd.isna(v):
            return default
        return int(v)
    except Exception:
        return default

def _list(v):
    """تحويل آمن إلى list من نصوص (يتعامل مع None و NaN و set و ndarray)."""
    if v is None:
        return []
    if isinstance(v, str):
        return [v]
    try:
        if not hasattr(v, "__iter__") and pd.isna(v):
            return []
    except Exception:
        pass
    try:
        return [str(_py(x)) for x in v]
    except TypeError:
        return []

شرح النتيجة (Score Explanation - AC11)


In [ ]:
FACTOR_NAMES = {
    "EN": {"required": "Required Skills", "experience": "Experience",
           "availability": "Availability", "quality": "Profile Quality",
           "preferred": "Preferred Skills"},
    "AR": {"required": "المهارات المطلوبة", "experience": "الخبرة",
           "availability": "التوافر", "quality": "جودة الملف",
           "preferred": "المهارات المفضلة"},
}

def generate_score_explanation(row, project, lang="AR"):
    lang = lang if lang in ("AR", "EN") else "AR"

    req_total = len(_list(project["required_skills"]))
    matched_req = len(_list(row.get("matched_skills")))
    years = _num(row.get("years"))
    completion = _num(row.get("completion_pct"))
    portfolio = _num(row.get("portfolio"))

    # تحديد حالة التوافر
    if bool(row.get("unavailable", False)):
        avail_en, avail_ar = "Unavailable", "غير متاح"
    elif bool(row.get("at_limit", False)):
        avail_en, avail_ar = "At the limit of projects", "وصل للحد الأقصى للمشاريع"
    else:
        avail_en, avail_ar = "Available now", "متاح الآن"

    texts = {
        "EN": {
            "required": f"Covers {matched_req} of {req_total} required skills",
            "experience": f"Has {years} years of experience",
            "availability": avail_en,
            "quality": f"Profile is {completion}% complete with {portfolio} portfolio items",
            "preferred": "Covers some preferred skills",
            "note": "This is a suggestion and not a guarantee of quality",
        },
        "AR": {
            "required": f"يغطي {matched_req} من أصل {req_total} مهارات مطلوبة",
            "experience": f"لديه {years} سنوات من الخبرة",
            "availability": avail_ar,
            "quality": f"الملف مكتمل بنسبة {completion}% ولديه {portfolio} أعمال",
            "preferred": "يغطي بعض المهارات المفضلة",
            "note": "هذا اقتراح وليس ضمانًا للجودة",
        },
    }
    t, names = texts[lang], FACTOR_NAMES[lang]

    factors = [
        {"factor": names["required"],     "points": _num(row.get("points_required")),     "text": t["required"]},
        {"factor": names["experience"],   "points": _num(row.get("points_experience")),   "text": t["experience"]},
        {"factor": names["availability"], "points": _num(row.get("points_availability")), "text": t["availability"]},
        {"factor": names["quality"],      "points": _num(row.get("points_quality")),      "text": t["quality"]},
    ]

    pref_points = _num(row.get("points_preferred"))
    if pref_points > 0:
        factors.append({"factor": names["preferred"], "points": pref_points, "text": t["preferred"]})

    return {
        "factors": factors,
        "missing_skills": _list(row.get("missing_skills")),
        "note": t["note"],
    }

حالة التحديث (Update Status & Refresh Logic - AC15, AC16)


In [ ]:
def get_refresh_status(last_update_time=None, current_time=None):
    if current_time is None:
        current_time = datetime.now()

    # AC16: الحساب لأول مرة
    if last_update_time is None:
        return {
            "can_refresh": False,
            "status_text_en": "We are preparing your matches...",
            "status_text_ar": "نقوم بتجهيز المطابقات...",
            "error_en": None,
            "error_ar": None,
        }

    minutes_passed = (current_time - last_update_time).total_seconds() / 60
    stamp = last_update_time.strftime("%H:%M")

    # AC15: منع التحديث خلال 10 دقائق من آخر تحديث
    if minutes_passed < 10:
        wait_time = max(1, math.ceil(10 - minutes_passed))  # تقريب للأعلى
        return {
            "can_refresh": False,
            "status_text_en": f"Updated {stamp}",
            "status_text_ar": f"آخر تحديث {stamp}",
            "error_en": f"You can refresh again in {wait_time} minutes",
            "error_ar": f"يمكنك التحديث مرة أخرى بعد {wait_time} دقيقة",
        }

    return {
        "can_refresh": True,
        "status_text_en": f"Updated {stamp}",
        "status_text_ar": f"آخر تحديث {stamp}",
        "error_en": None,
        "error_ar": None,
    }

تجهيز المخرجات للواجهة الأمامية (Format Output for Frontend / API Response)


In [ ]:
def prepare_matches_api_response(matches_df, project, page=1, lang="AR", last_update_time=None):
    refresh_info = get_refresh_status(last_update_time)

    # AC17: الحالات الفارغة (Empty State)
    if matches_df is None or len(matches_df) == 0:
        return {
            "refresh_info": refresh_info,
            "empty_state": True,
            "message_en": "No developers or teams match your project yet. Try changing the required skills",
            "message_ar": "لا يوجد مطورون أو فرق مطابقون لمشروعك حاليًا. جرّب تغيير المهارات المطلوبة",
            "data": [],
        }

    total_items = len(matches_df)
    total_pages = max(1, (total_items - 1) // PAGE_SIZE + 1)
    page = min(max(1, int(page)), total_pages)  # حماية من رقم صفحة خارج النطاق

    page_data = get_page(matches_df, page=page)

    results = []
    for _, row in page_data.iterrows():
        results.append({
            "rank": _num(row.get("rank")),
            "candidate_id": _py(row.get("candidate_id")),
            "type": _py(row.get("candidate_type")),
            "score": _num(row.get("score")),
            "matched_skills": _list(row.get("matched_skills")),
            "missing_skills": _list(row.get("missing_skills")),
            "marks": _list(row.get("marks")),
            "can_invite": bool(row.get("can_invite", False)),
            "score_explanation": generate_score_explanation(row, project, lang),
        })

    return {
        "refresh_info": refresh_info,
        "empty_state": False,
        "pagination": {
            "current_page": page,
            "items_per_page": PAGE_SIZE,
            "total_items": total_items,
            "total_pages": total_pages,
        },
        "filters": available_filters(matches_df),
        "data": results,
    }

تجربة المخرجات النهائية بصيغة JSON


In [ ]:
# تحديد مشروع تجريبي (demo) من البيانات
candidates_demo = projects_tech[
    (projects_tech["category"] == "Full Stack Development")
    & (projects_tech["n_skills"] >= 5)
    & (projects_tech["accepts"] == "both")
]
candidates_demo = candidates_demo[candidates_demo.apply(skills_in_pool, axis=1)]
assert len(candidates_demo) > 0, "لا يوجد مشروع تجريبي مناسب في البيانات"
demo = candidates_demo.iloc[0]

# حساب المطابقات لهذا المشروع
matches = rank_matches(demo, candidates, interactions)

# تم التحديث منذ 5 دقائق لتجربة رسالة الخطأ
last_update = datetime.now() - timedelta(minutes=5)

api_response = prepare_matches_api_response(
    matches_df=matches,
    project=demo,
    page=1,
    lang="AR",
    last_update_time=last_update,
)

# default=str يحمي من أي نوع غير قابل للتحويل إلى JSON
print(json.dumps(api_response, indent=2, ensure_ascii=False, default=str))